In [1]:
# train_supervised.py

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from chess import pgn
from tqdm import tqdm
import pickle

# Import the upgraded modules
from auxiliary_func_v2 import create_input_for_nn_v2, encode_moves
from dataset_v2 import ChessDatasetV2
from model_v2 import ChessModelV2

# --- Configuration ---
CONFIG = {
    'data_path': 'C:/Users/siegk/chess-engine-main/data/png', # Adjust if necessary
    'max_games_to_load': 50000,
    'batch_size': 128,
    'epochs': 20,
    'learning_rate': 1e-4,
    'value_loss_weight': 0.5, # The lambda for balancing policy and value loss
    'model_save_path': '../../models/ChessModelV2.pth',
    'map_save_path': '../../models/move_map_v2.pkl',
}

In [2]:
def load_games(data_path: str, max_games: int) -> list:
    """Loads PGN files up to a maximum number of games."""
    games = []
    pgn_files = [f for f in os.listdir(data_path) if f.endswith(".pgn")]
    for file_name in tqdm(pgn_files, desc="Loading PGN files"):
        if len(games) >= max_games:
            break
        with open(os.path.join(data_path, file_name), 'r') as pgn_file:
            while len(games) < max_games:
                game = pgn.read_game(pgn_file)
                if game is None:
                    break
                games.append(game)
    return games

In [3]:
print("Loading games...")
games = load_games(CONFIG['data_path'], CONFIG['max_games_to_load'])
print(f"Loaded {len(games)} games.")

Loading games...


Loading PGN files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading PGN files:  11%|█         | 1/9 [01:41<13:35, 101.92s/it]

Loaded 50000 games.


In [4]:
print("Creating training data...")
X, y_policy, z_value = create_input_for_nn_v2(games)

Creating training data...


In [5]:
print("Encoding moves...")
y_policy_encoded, move_to_int, int_to_move = encode_moves(y_policy)
num_classes = len(move_to_int)

Encoding moves...


In [6]:
# Save the move map for prediction later
with open(CONFIG['map_save_path'], 'wb') as f:
    pickle.dump({'move_to_int': move_to_int, 'int_to_move': int_to_move}, f)
print(f"Move map saved to {CONFIG['map_save_path']}")

Move map saved to ../../models/move_map_v2.pkl


In [ ]:
# --- 2. Setup for Training ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset = ChessDatasetV2(X, y_policy_encoded, z_value)
dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)

model = ChessModelV2(num_classes=num_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
    
policy_loss_fn = nn.CrossEntropyLoss()
value_loss_fn = nn.MSELoss()

In [14]:
# --- 3. Training Loop ---
print("Starting supervised training...")
for epoch in range(CONFIG['epochs']):
    model.train()
    total_loss = 0.0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{CONFIG['epochs']}")
        
    for board_states, policy_targets, value_targets in pbar:
        board_states = board_states.to(device)
        policy_targets = policy_targets.to(device)
        value_targets = value_targets.to(device).unsqueeze(1)

        optimizer.zero_grad()
            
        # Forward pass
        policy_logits, value_preds = model(board_states)
            
        # Calculate losses
        policy_loss = policy_loss_fn(policy_logits, policy_targets)
        value_loss = value_loss_fn(value_preds, value_targets)
            
        # Combined loss
        loss = policy_loss + CONFIG['value_loss_weight'] * value_loss
            
        loss.backward()
        optimizer.step()
            
        total_loss += loss.item()
        pbar.set_postfix({
            'Loss': f"{loss.item():.4f}",
            'P_Loss': f"{policy_loss.item():.4f}",
            'V_Loss': f"{value_loss.item():.4f}"
        })

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch + 1} finished. Average Loss: {avg_loss:.4f}")

Starting supervised training...


Epoch 1/20: 100%|██████████| 34142/34142 [16:37<00:00, 34.24it/s, Loss=4.0012, P_Loss=3.7318, V_Loss=0.5387]  


Epoch 1 finished. Average Loss: 3.0831


Epoch 2/20: 100%|██████████| 34142/34142 [19:19<00:00, 29.45it/s, Loss=2.2439, P_Loss=1.9388, V_Loss=0.6103]  


Epoch 2 finished. Average Loss: 2.4266


...

Epoch 18 finished. Average Loss: nan


Epoch 19/20: 100%|██████████| 34142/34142 [18:34<00:00, 30.62it/s, Loss=nan, P_Loss=nan, V_Loss=nan]


Epoch 19 finished. Average Loss: nan


Epoch 20/20: 100%|██████████| 34142/34142 [17:07<00:00, 33.23it/s, Loss=nan, P_Loss=nan, V_Loss=nan]

Epoch 20 finished. Average Loss: nan


In [15]:
# --- 4. Save the Final Model ---
torch.save(model.state_dict(), CONFIG['model_save_path'])
print(f"Training complete. Model saved to {CONFIG['model_save_path']}")

Training complete. Model saved to ../../models/ChessModelV2.pth


In [16]:
import pickle

with open("../../models/heavy_move_to_int2", "wb") as file:
    pickle.dump(move_to_int, file)